In [ ]:
# Install Required Libraries
!pip install einops albumentations torchinfo

In [ ]:
# Import Libraries
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [ ]:
!pip install ninja

In [ ]:
#Step 2 — C++ Header (swish.cpp)
%%writefile swish.cpp

#include <torch/extension.h>

torch::Tensor swish_cuda(torch::Tensor input);

torch::Tensor swish(torch::Tensor input)
{
    return swish_cuda(input);
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m)
{
    m.def("forward", &swish, "Swish Forward");
}

Overwriting swish.cpp


In [ ]:
#Step 3 — CUDA Kernel
%%writefile swish_kernel.cu

#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__
void swish_kernel(
    const float* input,
    float* output,
    int size)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if(idx < size)
    {
        float x = input[idx];

        output[idx] = x / (1.0f + expf(-x));
    }
}

torch::Tensor swish_cuda(torch::Tensor input)
{
    auto output = torch::zeros_like(input);

    int size = input.numel();

    int threads = 256;

    int blocks = (size + threads - 1) / threads;

    swish_kernel<<<blocks,threads>>>(
        input.data_ptr<float>(),
        output.data_ptr<float>(),
        size
    );

    return output;
}

Overwriting swish_kernel.cu


In [ ]:
#Step 4 — Compile Extension
from torch.utils.cpp_extension import load

swish_cuda = load(
    name="swish_cuda",
    sources=[
        "swish.cpp",
        "swish_kernel.cu"
    ],
    verbose=True
)

In [ ]:
#Step 5 — Test
import torch

device = "cuda"

x = torch.randn(10, device=device)

y = swish_cuda.forward(x)

print(x)

print(y)

tensor([ 1.8125, -0.1179, -1.0701,  0.1598,  0.6516,  1.1257, -0.8036,  1.8496,
        -1.1646, -0.6541], device='cuda:0')
tensor([ 1.5582, -0.0555, -0.2733,  0.0863,  0.4284,  0.8499, -0.2485,  1.5982,
        -0.2770, -0.2238], device='cuda:0')


In [ ]:
#Step 6 — Compare with PyTorch Swish
torch_swish = x * torch.sigmoid(x)

print(torch.allclose(
    y,
    torch_swish,
    atol=1e-6
))

True


In [ ]:
#Step 7 — Benchmark
import time

x = torch.randn(10000000, device="cuda")

torch.cuda.synchronize()

start = time.time()

for _ in range(100):
    y = swish_cuda.forward(x)

torch.cuda.synchronize()

custom_time = time.time() - start

print("Custom CUDA")

print(custom_time)

Custom CUDA
0.0571141242980957


In [ ]:
#Step 8 — Benchmark PyTorch
torch.cuda.synchronize()

start = time.time()

for _ in range(100):
    y = x * torch.sigmoid(x)

torch.cuda.synchronize()

torch_time = time.time() - start

print("PyTorch")

print(torch_time)

PyTorch
0.08331584930419922


In [ ]:
#Step 9 — Speed Comparison
print("Speedup")

print(torch_time/custom_time)

Speedup
1.4587608639388197
